**DSCI 552 Homework 3**
===================

- **Name:** Rudrani Chavarkar
- **GitHub Username:** rudranichav
- **USD ID:** 3158-2107-65

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [35]:
import numpy as np
import pandas as pd
import os
from glob import glob

Get the AReM Data Set

In [36]:
# Base directory
root_path = os.path.join("../data/AReM")

activity_list = [
    "bending1", "bending2",
    "cycling", "lying",
    "sitting", "standing", "walking"
]

### (b) Test and Train Data

* During preprocessing, I discovered that **dataset 4 in the bending2 class** contained only NaN values in part (c)(ii). This caused all extracted features in part (c)(iii) to also become NaN.

* Instead of deleting the dataset (which would remove useful information and potentially create class imbalance), I chose to investigate the issue.

* After opening the CSV file in Excel, I found that all values were incorrectly stored in the time column rather than being separated into their proper feature columns. Because of this formatting error, the cleaning step (`df.dropna(axis=1)`) removed all feature columns, resulting in empty data and NaN feature outputs.

* I corrected the file using **Data → Text to Columns** in Excel with Tab and Space as delimiters to properly split the values. After fixing the formatting, I reran the entire pipeline from the beginning, and the issue was resolved.

In [37]:
# Data Cleaning Function
def preprocess_df(dataframe):
    dataframe = dataframe.dropna(how="all")
    dataframe = dataframe.dropna(axis=1, how="all")
    dataframe = dataframe.apply(pd.to_numeric, errors="coerce")
    dataframe = dataframe.ffill().bfill()
    return dataframe


train_collection = []
test_collection = []

In [38]:
# Load and Split Data
for act in activity_list:

    act_path = os.path.join(root_path, act)

    if not os.path.isdir(act_path):
        print(f"Directory not found: {act_path}")
        continue

    # Determine test files
    if act in ("bending1", "bending2"):
        test_files_set = {"dataset1.csv", "dataset2.csv"}
    else:
        test_files_set = {"dataset1.csv", "dataset2.csv", "dataset3.csv"}

    file_paths = glob(os.path.join(act_path, "*.csv"))

    if len(file_paths) == 0:
        print(f"No CSV files inside {act_path}")
        continue

    for file_path in file_paths:

        filename = os.path.basename(file_path).lower()

        try:
            temp_df = pd.read_csv(file_path, skiprows=4, on_bad_lines="skip")
        except Exception as err:
            print(f"Error loading {file_path}: {err}")
            continue

        temp_df = preprocess_df(temp_df)
        temp_df["activity"] = act

        if filename in test_files_set:
            test_collection.append(temp_df)
        else:
            train_collection.append(temp_df)

In [39]:

# Safety Checks
if len(train_collection) == 0:
    raise RuntimeError("Training set is empty. Check directory names or split rule.")

if len(test_collection) == 0:
    raise RuntimeError("Testing set is empty. Check directory names or split rule.")

In [40]:
# Concatenate for inspection
train_full = pd.concat(train_collection, ignore_index=True)
test_full = pd.concat(test_collection, ignore_index=True)

print("Train shape:", train_full.shape)
print("Test shape:", test_full.shape)

Train shape: (33117, 8)
Test shape: (9120, 8)


In [41]:
print("\nTrain columns:")
train_full.head()



Train columns:


,# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23,activity
0,0.0,42.00,0.71,21.25,0.43,30.00,0.00,bending1
1,250.0,41.50,0.50,20.25,1.48,31.25,1.09,bending1
2,500.0,41.50,0.50,14.25,1.92,33.00,0.00,bending1
3,750.0,40.75,0.83,15.75,0.43,33.00,0.00,bending1
4,1000.0,40.00,0.71,20.00,2.74,32.75,0.43,bending1


In [42]:
print("Test columns:")
test_full.head()

Test columns:


,# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23,activity
0,0,39.25,0.43,22.75,0.43,33.75,1.3,bending1
1,250,39.25,0.43,23.00,0.00,33.00,0.0,bending1
2,500,39.25,0.43,23.25,0.43,33.00,0.0,bending1
3,750,39.50,0.50,23.00,0.71,33.00,0.0,bending1
4,1000,39.50,0.50,24.00,0.00,33.00,0.0,bending1


### (c) Feature Extraction

#### i. Research

### a) Statistical Features

* **Mean / Median:** Represent the central tendency or typical value of the time series.
* **Minimum / Maximum:** Capture the lowest and highest observed values in the signal.
* **Range:** Measures the spread between the minimum and maximum values.
* **Standard Deviation / Variance:** Quantify how much the data varies around the mean.
* **Quartiles & Interquartile Range (IQR):** Describe how values are distributed across percentiles and measure the spread of the middle 50% of the data.
* **Skewness:** Indicates whether the distribution is asymmetric around the mean.
* **Kurtosis:** Reflects the heaviness of the tails and the presence of extreme values.

---

### b) Time-Domain Features

* **Autocorrelation:** Evaluates the relationship between current values and previous values in the series.
* **Partial Autocorrelation:** Measures correlation at a given lag while removing the influence of intermediate lags.
* **Slope:** Represents the overall upward or downward trend over time.
* **Number of Peaks / Valleys:** Counts local maxima and minima within the signal.
* **Longest Consecutive Run:** Identifies extended periods where the signal remains stable or similar.
* **Zero-Crossing Rate:** Counts how frequently the signal changes sign.
* **Energy:** Measures signal strength using the sum of squared values.
* **Entropy:** Assesses the level of randomness or unpredictability in the signal.
* **Root Mean Square (RMS):** Provides an overall measure of signal magnitude.
* **Mean Absolute Deviation:** Calculates the average absolute difference from the mean.

---

### c) Frequency-Domain Features

* **Spectral Density:** Describes how signal energy is distributed across different frequency components.
* **Peak Frequencies:** Identify the dominant or most significant frequencies present in the signal.

---

### d) Decomposition-Based Features

* **Trend:** Represents the long-term movement or direction in the time series.
* **Seasonality:** Captures recurring patterns at fixed intervals.
* **Residual:** Represents the remaining irregular or random variation after removing trend and seasonal components.


#### ii. Extraction

In [43]:
# Feature Extraction Function
def compute_features(frame):
    result = {}

    # Skip first column (time) and last column (activity)
    for idx, column in enumerate(frame.columns[1:-1]):
        series_data = frame[column].astype(float)

        result[f"min{idx+1}"] = series_data.min()
        result[f"max{idx+1}"] = series_data.max()
        result[f"mean{idx+1}"] = series_data.mean()
        result[f"median{idx+1}"] = series_data.median()
        result[f"std{idx+1}"] = series_data.std()
        result[f"1st quart{idx+1}"] = series_data.quantile(0.25)
        result[f"3rd quart{idx+1}"] = series_data.quantile(0.75)

    return result


train_feature_list = []
test_feature_list = []

for frame in train_collection:
    train_feature_list.append(compute_features(frame))

for frame in test_collection:
    test_feature_list.append(compute_features(frame))


train_features_df = pd.DataFrame(train_feature_list)
test_features_df = pd.DataFrame(test_feature_list)

train_features_df.insert(0, "Instance", range(1, len(train_features_df) + 1))
test_features_df.insert(0, "Instance", range(1, len(test_features_df) + 1))

train_features_df.head()

,Instance,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
0,1,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,2,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,3,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,4,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,5,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


To construct the table in the required format, I combined the training and testing feature DataFrames into a single final dataset. This merged DataFrame contains all 88 instances (one per file across all activity classes) and a total of 43 columns: one **Instance** column plus 42 feature columns (6 time series × 7 extracted features per series).

In [44]:
# Combine datasets
combined_dataset = pd.concat([train_features_df, test_features_df], ignore_index=True)
combined_dataset["Instance"] = range(1, len(combined_dataset) + 1)

combined_dataset.head()

,Instance,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
0,1,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,2,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,3,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,4,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,5,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


#### iii. Standard Deviation

In [45]:
# Bootstrap Standard Deviation CI
def bootstrap_std_confidence(arr, iterations=1000, percentile_range=(5, 95)):

    arr = pd.to_numeric(arr, errors="coerce").dropna().values
    sample_size = len(arr)

    boot_values = [
        np.std(
            np.random.choice(arr, size=sample_size, replace=True),
            ddof=1
        )
        for _ in range(iterations)
    ]

    avg_std = np.mean(boot_values)
    conf_interval = np.percentile(boot_values, percentile_range)

    return avg_std, conf_interval


feature_only = combined_dataset.drop(columns="Instance")
bootstrap_output = {}

for feat in feature_only.columns:

    estimated_std, ci_bounds = bootstrap_std_confidence(
        feature_only[feat],
        iterations=1000,
        percentile_range=(5, 95)
    )

    bootstrap_output[feat] = {
        "bootstrap std estimate": estimated_std,
        "90% CI": ci_bounds
    }


bootstrap_summary_df = pd.DataFrame.from_dict(
    bootstrap_output,
    orient="index"
)

bootstrap_summary_df.head()

,bootstrap std estimate,90% CI
min1,9.543479,"[8.28105686658373, 10.844586539965325]"
max1,4.135553,"[3.0912533388638113, 5.1312081101055]"
mean1,5.220293,"[4.643052127024556, 5.823971141759408]"
median1,5.344625,"[4.693627571374211, 5.927067241561765]"
std1,1.752497,"[1.577823859808374, 1.9501227214260684]"


#### iv. Select Features

From the time-domain features extracted in part c(ii), I consider the minimum, median, and maximum to be the most informative.

**a) Minimum (Min):**
The minimum value captures the lowest sensor reading in a time series. In the context of the AReM dataset, this can reflect periods when the subject is relatively inactive or at rest, as sensor readings are typically low during minimal or no movement.

**b) Median:**
The median represents the middle value of the dataset once sorted, indicating the point at which half of the observations lie above and half below. I selected the median over the mean because real-world sensor data often contains outliers caused by sudden movements, noise, or sensor errors. These outliers can skew the mean, making it less reliable, whereas the median is robust to such extremes and provides a more stable measure of typical sensor activity.

**c) Maximum (Max):**
The maximum value identifies the highest sensor reading in the time series. For the AReM dataset, this corresponds to moments of peak activity, when movement intensity is greatest and sensor readings reach their highest levels.


## 2. ISLR 3.7.4

### (a) Linear Train

When the true relationship between 𝑋 and 𝑌 is linear, a linear model is well suited to the data and can fit it effectively during training. Since there are no complex non-linear patterns to capture, the linear model focuses on the underlying relationship and typically achieves a low training RSS.

In contrast, a cubic model, being more flexible, may attempt to fit unnecessary complexity in the data. Its ability to capture higher-order patterns means it can also fit random noise if present. Consequently, the cubic model might slightly overfit the training data. As a result, while the cubic model may have a similar or slightly lower training RSS compared to the linear model, this reduction often reflects noise fitting rather than capturing meaningful structure.

### (b) Linear Test

For a genuinely linear relationship, the linear model typically generalizes better to test data. Because it avoids overfitting, it captures the true underlying pattern and performs consistently on unseen observations.

Although a cubic model may fit the training data well, it carries a higher risk of overfitting by capturing noise that does not generalize. Consequently, its performance on test data can decline. Therefore, in most cases, the cubic model will exhibit a test RSS that is similar to or higher than that of the linear model.

### (c) Not Linear Train

When the relationship between 𝑋 and 𝑌 is non-linear, a linear model cannot adequately capture the true structure of the data, resulting in underfitting and a higher training RSS. In contrast, a cubic model, with its additional polynomial terms, is better suited to modeling non-linear patterns.

Consequently, the cubic model fits the training data more closely, achieving a lower training RSS than the linear model. However, this added flexibility also increases the risk of fitting noise in the data.

### (d) Not Linear Testing

For a non-linear relationship, a cubic model is generally expected to perform better on test data. Its capacity to capture curvature and complex patterns enables it to generalize more effectively to unseen observations, whereas a linear model is too restrictive.

As a result, the cubic model typically achieves a lower test RSS than the linear model, provided that overfitting remains limited.

## 3. ISLR 3.7.3 - Extra Practice 

The model equation is:

SS = 50 + 20(GPA) + 0.07(IQ) + 35(LEVEL) + 0.01(GPA × IQ) − 10(GPA × LEVEL)

a) For college students (LEVEL = 1):

SS_col = 50 + 20(GPA) + 0.07(IQ) + 35(1) + 0.01(GPA × IQ) − 10(GPA × 1)

For high school students (LEVEL = 0):

SS_hs = 50 + 20(GPA) + 0.07(IQ) + 35(0) + 0.01(GPA × IQ) − 10(GPA × 0)

Difference:

DIFFSS = SS_col − SS_hs = 35 − 10(GPA)

i) GPA > 3.5: DIFFSS < 0 → High school students earn more.

ii) GPA < 3.5: DIFFSS > 0 → College students earn more.

Therefore, the correct option is (iii).

In [46]:
# b) Calculate starting salary for a college graduate
GPA = 4.0
IQ = 110
LEVEL = 1  # College student

SScol = 50 + 20*GPA + 0.07*IQ + 35*LEVEL + 0.01*(GPA*IQ) - 10*(GPA*LEVEL)
print(f"The starting salary of a college graduate with IQ {IQ} and GPA {GPA} is {SScol}")

The starting salary of a college graduate with IQ 110 and GPA 4.0 is 137.1


c) False. The relevance of an interaction term is not determined solely by its coefficient value. Factors such as the variation in the interaction term, as well as statistical measures like standard errors and p-values, play a crucial role in determining its significance and explanatory power.